# $B^\pm\to K^\pm\pi^+\pi^-$ CP fit with efficiency and background

A simultaneous direct-CP amplitude fit with charge-symmetric detector efficiency and background. The independent Dalitz variables are $s_{13}=m^2(K^\pm\pi^\mp)$ and $s_{23}=m^2(\pi^+\pi^-)$.


In [ ]:
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dalitzplotfitter import (
    BaBarFlatte, CPRealImag, DecayChannel, DecayModel, LASS, Minimizer,
    NonResonant, Parameter, PhaseSpaceSample, Resonance, enable_x64,
    weighted_resample,
)
from dalitzplotfitter.background import FunctionalBackground
from dalitzplotfitter.efficiency import FunctionalEfficiency

enable_x64()


## 1. Shared CP amplitude model


In [ ]:
truth_spec = {
    "Kstar892": (1.00, 0.00, +0.04, -0.03),
    "KpiS":     (1.40, -0.60, -0.10, +0.08),
    "rho770":   (0.65, 0.10, +0.06, +0.04),
    "f0_980":   (-0.20, 1.00, -0.05, +0.07),
    "NR":       (-0.50, 0.10, 0.00, 0.00),
}
truth = {}
shared = {}
for name, (x, y, dx, dy) in truth_spec.items():
    pars = (
        Parameter.coefficient(f"{name}.x", x, owner=name, fixed=(name == "Kstar892"), step=0.01),
        Parameter.coefficient(f"{name}.y", y, owner=name, fixed=(name == "Kstar892"), step=0.01),
        Parameter.coefficient(f"{name}.dx", dx, owner=name, fixed=(name == "NR"), step=0.01),
        Parameter.coefficient(f"{name}.dy", dy, owner=name, fixed=(name == "NR"), step=0.01),
    )
    shared[name] = CPRealImag(*pars)
    truth.update({p.name: p.value for p in pars})

def components(charge):
    c = {name: value.for_charge(charge) for name, value in shared.items()}
    return [
        Resonance("Kstar892", (0,2), c["Kstar892"], mass=0.8958, width=0.0474, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("KpiS", (0,2), c["KpiS"], lineshape=LASS(2.07, 3.32, 1.8), mass=1.425, width=0.270, spin=0, resonance_radius=4.0, parent_radius=4.0),
        Resonance("rho770", (1,2), c["rho770"], mass=0.7753, width=0.1491, spin=1, resonance_radius=4.0, parent_radius=4.0),
        Resonance("f0_980", (1,2), c["f0_980"], lineshape=BaBarFlatte(), mass=0.965, width=0.0, spin=0, resonance_radius=4.0, parent_radius=4.0),
        NonResonant(c["NR"]),
    ]

plus_model = DecayModel(DecayChannel("B+", ("K+", "pi+", "pi-")), components(+1), normalization_method="square-dalitz", normalization_resolution=350, normalization_pair=(0,2))
minus_model = DecayModel(DecayChannel("B-", ("K-", "pi-", "pi+")), components(-1), normalization_method="square-dalitz", normalization_resolution=350, normalization_pair=(0,2))
plus_norm, minus_norm = plus_model.normalization_sample, minus_model.normalization_sample


## 2. Efficiency and background in $s_{13}$-$s_{23}$


In [ ]:
mK, mpi, _ = plus_model.channel.daughter_masses
mB = plus_model.channel.parent_mass
s13_min, s13_max = (mK + mpi)**2, (mB - mpi)**2
s23_min, s23_max = (2*mpi)**2, (mB - mK)**2

def scaled(data, key, low, high):
    return jnp.clip((data[key]-low)/(high-low), 0.0, 1.0)
efficiency = FunctionalEfficiency(lambda data: 0.55 + 0.30*scaled(data, "s13", s13_min, s13_max) + 0.10*jnp.cos(jnp.pi*scaled(data, "s23", s23_min, s23_max)))
background = FunctionalBackground(lambda data: 0.50 + 1.20*scaled(data, "s13", s13_min, s13_max) + 0.40*scaled(data, "s23", s23_min, s23_max))
eff_plus_norm = efficiency(plus_norm.as_dict())
eff_minus_norm = efficiency(minus_norm.as_dict())
bkg_plus_norm = jnp.mean(plus_norm.weights * background(plus_norm.as_dict()))
bkg_minus_norm = jnp.mean(minus_norm.weights * background(minus_norm.as_dict()))


## 3. Generate signal plus background for both charges


In [ ]:
N_POOL = 250_000
N_DATA = 50_000
BACKGROUND_FRACTION_TRUE = 0.18
plus_pool = plus_model.generate_phase_space(N_POOL, seed=6001)
minus_pool = minus_model.generate_phase_space(N_POOL, seed=6002)
plus_pool_cache = plus_model.prepare_cache(plus_pool, plus_norm, efficiency_normalization=eff_plus_norm)
minus_pool_cache = minus_model.prepare_cache(minus_pool, minus_norm, efficiency_normalization=eff_minus_norm)
splus = float(plus_pool_cache.normalization(truth))
sminus = float(minus_pool_cache.normalization(truth))
signal_pplus = splus/(splus+sminus)
rng = np.random.default_rng(6003)
n_background = rng.binomial(N_DATA, BACKGROUND_FRACTION_TRUE)
n_signal = N_DATA - n_background
n_signal_plus = rng.binomial(n_signal, signal_pplus)
n_signal_minus = n_signal - n_signal_plus
n_bkg_plus = rng.binomial(n_background, 0.5)
n_bkg_minus = n_background - n_bkg_plus
plus_signal = weighted_resample(jax.random.key(6004), plus_pool, plus_pool.weights * efficiency(plus_pool.as_dict()) * plus_pool_cache.intensity(truth), n_signal_plus, replace=True)
minus_signal = weighted_resample(jax.random.key(6005), minus_pool, minus_pool.weights * efficiency(minus_pool.as_dict()) * minus_pool_cache.intensity(truth), n_signal_minus, replace=True)
plus_bkg = weighted_resample(jax.random.key(6006), plus_pool, plus_pool.weights * background(plus_pool.as_dict()), n_bkg_plus, replace=True)
minus_bkg = weighted_resample(jax.random.key(6007), minus_pool, minus_pool.weights * background(minus_pool.as_dict()), n_bkg_minus, replace=True)
def merge(a, b):
    def joined(name):
        x, y = getattr(a, name), getattr(b, name)
        return None if x is None else jnp.concatenate((x, y))
    return PhaseSpaceSample(s12=joined("s12"), s13=joined("s13"), s23=joined("s23"), weights=jnp.ones((a.size+b.size,)), p1=joined("p1"), p2=joined("p2"), p3=joined("p3"))
plus_data = merge(plus_signal, plus_bkg)
minus_data = merge(minus_signal, minus_bkg)
print("B+ total:", plus_data.size, "B- total:", minus_data.size)


## 4. Joint CP mixture likelihood


In [ ]:
plus_cache = plus_model.prepare_cache(plus_data, plus_norm, efficiency_normalization=eff_plus_norm)
minus_cache = minus_model.prepare_cache(minus_data, minus_norm, efficiency_normalization=eff_minus_norm)
eff_plus_data = efficiency(plus_data.as_dict())
eff_minus_data = efficiency(minus_data.as_dict())
bkg_plus_data = background(plus_data.as_dict())
bkg_minus_data = background(minus_data.as_dict())
background_fraction = Parameter("background_fraction", 0.12, bounds=(0.001, 0.50), step=0.01)
fit_parameters = tuple(p for p in plus_model.parameters if not p.fixed) + (background_fraction,)
def nll(values):
    signal_norm_plus = plus_cache.normalization(values)
    signal_norm_minus = minus_cache.normalization(values)
    signal_norm_total = signal_norm_plus + signal_norm_minus
    signal_plus = eff_plus_data * plus_cache.intensity(values) / signal_norm_total
    signal_minus = eff_minus_data * minus_cache.intensity(values) / signal_norm_total
    background_norm_total = bkg_plus_norm + bkg_minus_norm
    bkg_plus = bkg_plus_data / background_norm_total
    bkg_minus = bkg_minus_data / background_norm_total
    f = values["background_fraction"]
    pdf_plus = (1.0-f)*signal_plus + f*bkg_plus
    pdf_minus = (1.0-f)*signal_minus + f*bkg_minus
    return -jnp.sum(jnp.log(jnp.clip(pdf_plus, min=1e-300))) - jnp.sum(jnp.log(jnp.clip(pdf_minus, min=1e-300)))

# Deliberately offset every floating parameter, including the background fraction.
rng = np.random.default_rng(6008)
start = {p.name: truth[p.name] + rng.normal(0.0, 0.08) for p in plus_model.parameters if not p.fixed}
start["background_fraction"] = float(np.clip(BACKGROUND_FRACTION_TRUE + rng.normal(0.0, 0.04), 0.01, 0.49))
result = Minimizer(nll, fit_parameters, verbose=1).fit(start_values=start, simplex=True, ncall=60_000)
fit_values = {p.name: float(result.values[p.name]) for p in plus_model.parameters if not p.fixed}
fit_values["background_fraction"] = float(result.values["background_fraction"])
print("valid:", result.valid, "NLL:", result.fval, "EDM:", result.fmin.edm)
print(f"{'parameter':20s} {'generated':>11s} {'start':>11s} {'fitted':>11s} {'error':>11s} {'pull':>9s}")
for p in plus_model.parameters:
    if p.fixed:
        continue
    fitted = float(result.values[p.name]); error = float(result.errors[p.name])
    print(f"{p.name:20s} {truth[p.name]:11.5f} {start[p.name]:11.5f} {fitted:11.5f} {error:11.5f} {(fitted-truth[p.name])/error:9.3f}")
bf_fit = float(result.values["background_fraction"]); bf_err = float(result.errors["background_fraction"])
print(f"{'background_fraction':20s} {BACKGROUND_FRACTION_TRUE:11.5f} {start['background_fraction']:11.5f} {bf_fit:11.5f} {bf_err:11.5f} {(bf_fit-BACKGROUND_FRACTION_TRUE)/bf_err:9.3f}")

print("\nB+ generated fit fractions (with efficiency):")
plus_model.print_fit_fractions(truth, normalization_sample=plus_norm, efficiency=efficiency, include_interference=True)
print("\nB+ fitted fit fractions (with efficiency):")
plus_model.print_fit_fractions(fit_values, normalization_sample=plus_norm, efficiency=efficiency, include_interference=True)
print("\nB- generated fit fractions (with efficiency):")
minus_model.print_fit_fractions(truth, normalization_sample=minus_norm, efficiency=efficiency, include_interference=True)
print("\nB- fitted fit fractions (with efficiency):")
minus_model.print_fit_fractions(fit_values, normalization_sample=minus_norm, efficiency=efficiency, include_interference=True)


## 5. Charge-separated Dalitz plots


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
for ax, sample, title in ((axes[0], plus_data, "B+"), (axes[1], minus_data, "B-")):
    h = ax.hist2d(np.asarray(sample.s13), np.asarray(sample.s23), bins=80)
    fig.colorbar(h[3], ax=ax, label="events")
    ax.set(xlabel=r"$s_{13}$ [GeV$^2$]", ylabel=r"$s_{23}$ [GeV$^2$]", title=title)
plt.show()


## 6. Final $s_{13}$ and $s_{23}$ projections

The projections below include efficiency and background and are shown separately for $B^+$ and $B^-$.


In [ ]:
eff_plus_pool = efficiency(plus_pool.as_dict())
eff_minus_pool = efficiency(minus_pool.as_dict())
bkg_plus_pool = background(plus_pool.as_dict())
bkg_minus_pool = background(minus_pool.as_dict())
def mixture_projection(pool, cache, eff_pool, bkg_pool, values, f_bkg, variable, bins):
    signal_norm_total = plus_pool_cache.normalization(values) + minus_pool_cache.normalization(values)
    background_norm_total = bkg_plus_norm + bkg_minus_norm
    signal_weights = pool.weights * eff_pool * cache.intensity(values) / signal_norm_total
    background_weights = pool.weights * bkg_pool / background_norm_total
    total_weights = (1.0-f_bkg)*signal_weights + f_bkg*background_weights
    hist, _ = np.histogram(np.asarray(getattr(pool, variable)), bins=bins, weights=np.asarray(total_weights))
    return hist
fig, axes = plt.subplots(2, 2, figsize=(13, 9), constrained_layout=True)
for row, (sample, pool, cache, eff_pool, bkg_pool, title) in enumerate(((plus_data, plus_pool, plus_pool_cache, eff_plus_pool, bkg_plus_pool, "B+"), (minus_data, minus_pool, minus_pool_cache, eff_minus_pool, bkg_minus_pool, "B-"))):
    for col, (variable, label) in enumerate((("s13", r"$s_{13}$ [GeV$^2$]"), ("s23", r"$s_{23}$ [GeV$^2$]"))):
        ax = axes[row, col]
        observed = np.asarray(getattr(sample, variable))
        bins = np.linspace(observed.min(), observed.max(), 70)
        centers = 0.5*(bins[:-1] + bins[1:])
        data_hist, _ = np.histogram(observed, bins=bins)
        truth_hist = mixture_projection(pool, cache, eff_pool, bkg_pool, truth, BACKGROUND_FRACTION_TRUE, variable, bins)
        fitted_hist = mixture_projection(pool, cache, eff_pool, bkg_pool, fit_values, fit_values["background_fraction"], variable, bins)
        truth_hist *= data_hist.sum()/truth_hist.sum()
        fitted_hist *= data_hist.sum()/fitted_hist.sum()
        ax.errorbar(centers, data_hist, yerr=np.sqrt(np.maximum(data_hist, 1.0)), fmt=".", label="toy data")
        ax.step(centers, truth_hist, where="mid", linestyle="--", label="generated model")
        ax.step(centers, fitted_hist, where="mid", label="fitted model")
        ax.set(xlabel=label, ylabel="events / bin", title=title)
        ax.legend()
plt.show()


## Interpretation

This example keeps charge as part of the fitted sample space while adding detector efficiency and a charge-symmetric background component. The fit starts from deliberately displaced parameter values and reports charge-separated generated and fitted fit fractions.
